In [ ]:
# ! pip install segmentation-models-pytorch albumentations scikit-learn

In [ ]:
import glob
import os
import random

import albumentations as A
import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

from losses import ComboLoss

In [ ]:
SEED = 60

def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

    os.environ['PYTHONHASHSEED'] = str(SEED)
    os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
    os.environ['CUDA_VISIBLE_DEVICES']= '1'

device = 'cuda'

BATCH_SIZE = 4

In [ ]:
def plot_images_side_by_side(image1, image2, title1='', title2=''):
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))
    
    axs[0].imshow(image1)
    axs[0].set_title(title1)
    axs[0].axis('off')
    
    axs[1].imshow(image2)
    axs[1].set_title(title2)
    axs[1].axis('off')
    
    plt.show()


def dice_channels(prob, truth, threshold=0.5, eps = 1E-9):
    num_imgs = prob.size(0)
    num_channels = prob.size(1)
    prob = (prob > threshold).float()
    truth = (truth > 0.5).float()
    prob = prob.view(num_imgs, num_channels, -1)
    truth = truth.view(num_imgs, num_channels, -1)
    intersection = (prob * truth)
    score = (2. * intersection.sum(2) + eps) / (prob.sum(2) + truth.sum(2) + eps)
    score[score >= 1] = 1
    return score.mean()


def train_epoch(loader, model, loss_function_seg, optimizer, device):
    model = model.to(device)
    model.train()
    avg_loss = 0.
    optimizer.zero_grad()
    for image, mask in loader:
        x = image.to(device)
        y = mask.to(device)
        prediction_seg = model(x)
        loss = loss_function_seg(prediction_seg, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad() 
        avg_loss += loss.item()
    avg_loss /= len(loader)
    return avg_loss


def valid_epoch(loader, model, device):
    model = model.to(device)
    model.eval()
    scores = []
    with torch.no_grad():
        for image, mask in loader:
            x = image.to(device)
            y = mask.to(device)
            probs = torch.sigmoid(model(x))
            scores.append(dice_channels(probs, y))
    return torch.stack(scores).mean().item()

In [ ]:
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Resize(384, 384),
    A.Normalize(),
])

test_transforms = A.Compose([
    A.Resize(384, 384),
    A.Normalize(),
])

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, img_paths, masks_paths, transforms):
        self.img_paths = img_paths
        self.masks_paths = masks_paths
        self.transforms = transforms
        
    def __getitem__(self, item):
        img_path = self.img_paths[item]
        mask_path = self.masks_paths[item]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = np.load(mask_path)
        
        transformed_data = self.transforms(image=img, mask=mask)
        img = transformed_data['image']
        mask = transformed_data['mask']

        return torch.from_numpy(img).permute(2, 0, 1), torch.from_numpy(mask).permute(2, 0, 1).float()
    
    def __len__(self):
        return len(self.img_paths)

In [ ]:
img_path = sorted(glob.glob('data/train/images/*.jpg'))[0]
mask_path = sorted(glob.glob('data/train/masks/*.npy'))[0]
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
mask = np.load(mask_path)

transformed_data = train_transforms(image=img, mask=mask)
transformed_img = transformed_data['image']
transformed_mask = transformed_data['mask']

In [ ]:
plot_images_side_by_side(img, mask[:, :, 0])

In [ ]:
plot_images_side_by_side(transformed_img, transformed_mask[:, :, 0])

In [ ]:
set_seed()

In [ ]:
all_train_images = sorted(glob.glob('data/train/images/*.jpg'))
all_train_masks = sorted(glob.glob('data/train/masks/*.npy'))

train_images, valid_images, train_masks, valid_masks = train_test_split(
    all_train_images,
    all_train_masks,
    test_size=0.2,
    random_state=42
)

all_test_images = sorted(glob.glob('data/test/images/*.jpg'))

len(train_images), len(train_masks), len(valid_images), len(valid_masks), len(all_test_images)

In [ ]:
train_dataset = SegmentationDataset(
    train_images, 
    train_masks,
    train_transforms
)

valid_dataset = SegmentationDataset(
    valid_images, 
    valid_masks,
    test_transforms
)

train_loader = DataLoader(train_dataset, BATCH_SIZE, num_workers=4, shuffle=True)
valid_loader = DataLoader(valid_dataset, BATCH_SIZE, num_workers=4, shuffle=False)

print(len(train_dataset), len(valid_dataset))

In [ ]:
sample = train_dataset[2]

In [ ]:
plot_images_side_by_side(sample[0].permute(1,2,0), sample[1].sum(0))

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    classes=4,
).to(device)

loss_fn = ComboLoss(weights={'bce': 1, 'dice': 0}, channel_weights=[1]*4)
optimizer = torch.optim.Adam(model.parameters(),  lr=5e-4)

In [ ]:
best_score = 0

for epoch in range(30):
    train_loss = train_epoch(train_loader, model, loss_fn, optimizer, device)
    valid_score = valid_epoch(valid_loader, model, device)
    
    if valid_score > best_score:
        torch.save(model.state_dict(), 'best_model.pth')
        best_score = valid_score
    print(f'Epoch: {epoch}, train_loss: {train_loss:.4f}, valid_score: {valid_score:.3f}\n')

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))

best_score = valid_epoch(valid_loader, model, device)
print(f'Best validation score: {best_score}')

In [ ]:
valid_image_path = sorted(valid_images)[7]

with torch.no_grad():
    img = cv2.imread(valid_image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transformed_data = test_transforms(image=img)
    transformed_img = transformed_data['image']
    model_input = torch.from_numpy(transformed_img).permute(2, 0, 1).unsqueeze(0)
    pred = model(model_input.to(device))
    mask = torch.sigmoid(pred).squeeze().cpu() > 0.5
    resized_mask = cv2.resize(
        mask.numpy().transpose((1, 2, 0)).astype(int),
        (img.shape[1], img.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )

In [ ]:
plot_images_side_by_side(img, resized_mask.sum(2, keepdims=True))